# Módulo 04 · Aula 05 — Ecossistema Avançado

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"O script rodou de madrugada e falhou. Não sei em que etapa, não sei com qual arquivo, não sei que hora. Os `print` foram todos para o nada, porque ninguém estava olhando o terminal."*

Três ferramentas resolvem isso: **datas** feitas direito, **logging** em vez de `print`, e **context managers** que garantem limpeza.

## O que você vai aprender aqui

| # | Tópico | Resolve |
|---|--------|---------|
| 1 | `datetime`, `date`, `timedelta` | Aritmética de datas sem gambiarra |
| 2 | Fusos horários (`zoneinfo`) | O bug que aparece só em produção |
| 3 | Parsing e formatação | Ler e escrever datas |
| 4 | **`logging`** | Substituir `print` por observabilidade |
| 5 | Handlers, formatters, níveis | Configuração profissional |
| 6 | Logging estruturado (JSON) | Logs que máquinas leem |
| 7 | **Context managers** | Garantir limpeza |
| 8 | `contextlib` | Atalhos essenciais |

## 1. `datetime` — os quatro tipos

| Tipo | Guarda | Exemplo |
|------|--------|---------|
| `date` | Ano, mês, dia | `2026-08-12` |
| `time` | Hora, minuto, segundo | `14:30:00` |
| `datetime` | Data **e** hora | `2026-08-12 14:30:00` |
| `timedelta` | Uma **duração** | `30 dias, 4:00:00` |

In [ ]:
from datetime import date, time, datetime, timedelta

hoje = date(2026, 8, 12)
agora = datetime(2026, 8, 12, 14, 30, 45)
duracao = timedelta(days=30, hours=4)

print(f"date     : {hoje}          tipo: {type(hoje).__name__}")
print(f"datetime : {agora}   tipo: {type(agora).__name__}")
print(f"timedelta: {duracao}   tipo: {type(duracao).__name__}")
print()
print("Componentes:", agora.year, agora.month, agora.day, agora.hour, agora.minute)
print("Dia da semana (0=seg):", agora.weekday())
print("ISO weekday (1=seg)  :", agora.isoweekday())
print("Dia do ano           :", agora.timetuple().tm_yday)

### Aritmética de datas

A regra é simples:

- `data ± timedelta` → **data**
- `data - data` → **timedelta**

In [ ]:
pedido_em = date(2026, 7, 15)
prazo = timedelta(days=10)

entrega_prevista = pedido_em + prazo
entregue_em = date(2026, 7, 28)
atraso = entregue_em - entrega_prevista

print(f"Pedido    : {pedido_em}")
print(f"Previsão  : {entrega_prevista}")
print(f"Entrega   : {entregue_em}")
print(f"Atraso    : {atraso.days} dias")
print()
print(f"Tipo de (data - data): {type(atraso).__name__}")
print(f"Em segundos: {atraso.total_seconds():,.0f}")

In [ ]:
# ⚠️ timedelta NÃO tem meses nem anos — e por um bom motivo
try:
    timedelta(months=1)
except TypeError as erro:
    print(f"❌ {erro}")

print("\n💭 Por quê? Porque 'um mês' é ambíguo:")
print("   31/01 + 1 mês = 28/02? 03/03? 29/02 em ano bissexto?")
print("   O Python se recusa a adivinhar.")
print()
print("Soluções:")
print("  • timedelta(days=30)              → aproximação explícita")
print("  • dateutil.relativedelta          → biblioteca externa, faz certo")
print("  • cálculo manual com ano/mês      → controle total")

In [ ]:
# Cálculo manual de "mês seguinte" — sem dependência externa
def adicionar_meses(d: date, meses: int) -> date:
    """Adiciona meses ajustando o dia quando necessário.

    31/01 + 1 mês = 28/02 (último dia válido do mês destino).
    """
    import calendar
    mes = d.month - 1 + meses
    ano = d.year + mes // 12
    mes = mes % 12 + 1
    dia = min(d.day, calendar.monthrange(ano, mes)[1])
    return date(ano, mes, dia)


for origem in [date(2026, 1, 31), date(2026, 3, 15), date(2026, 11, 30)]:
    print(f"{origem} + 1 mês = {adicionar_meses(origem, 1)}")
    print(f"{origem} + 3 meses = {adicionar_meses(origem, 3)}")

In [ ]:
# Períodos úteis para relatórios
def primeiro_dia_do_mes(d: date) -> date:
    return d.replace(day=1)


def ultimo_dia_do_mes(d: date) -> date:
    import calendar
    return d.replace(day=calendar.monthrange(d.year, d.month)[1])


def dias_uteis(inicio: date, fim: date) -> int:
    """Conta dias úteis (seg-sex). Não considera feriados."""
    total, atual = 0, inicio
    while atual <= fim:
        if atual.weekday() < 5:
            total += 1
        atual += timedelta(days=1)
    return total


ref = date(2026, 7, 15)
inicio, fim = primeiro_dia_do_mes(ref), ultimo_dia_do_mes(ref)

print(f"Referência : {ref}")
print(f"Início mês : {inicio}")
print(f"Fim do mês : {fim}")
print(f"Dias totais: {(fim - inicio).days + 1}")
print(f"Dias úteis : {dias_uteis(inicio, fim)}")

## 2. Fusos horários — o bug que só aparece em produção

Existem dois tipos de `datetime`:

| Tipo | Tem fuso? | Chamado de |
|------|-----------|------------|
| Sem `tzinfo` | ❌ | *naive* (ingênuo) |
| Com `tzinfo` | ✅ | *aware* (consciente) |

> 🔴 **`datetime.now()` devolve um datetime NAIVE** com a hora local da máquina. Em desenvolvimento você está em São Paulo; o servidor de produção está em UTC. Resultado: **3 horas de diferença** que ninguém percebe até um relatório sair errado na virada do mês.
>
> **Regra profissional:** guarde tudo em **UTC**, converta para o fuso local só na apresentação.

In [ ]:
from datetime import timezone
from zoneinfo import ZoneInfo          # biblioteca padrão desde o Python 3.9

# ❌ Naive: não sabe em que fuso está
naive = datetime(2026, 8, 12, 14, 30)
print(f"naive : {naive}   tzinfo={naive.tzinfo}")

# ✅ Aware: sabe exatamente qual instante representa
utc = datetime(2026, 8, 12, 17, 30, tzinfo=timezone.utc)
sp = utc.astimezone(ZoneInfo("America/Sao_Paulo"))
tokyo = utc.astimezone(ZoneInfo("Asia/Tokyo"))
ny = utc.astimezone(ZoneInfo("America/New_York"))

print(f"\nO MESMO instante em fusos diferentes:")
print(f"  UTC       : {utc}")
print(f"  São Paulo : {sp}")
print(f"  Tóquio    : {tokyo}")
print(f"  Nova York : {ny}")
print(f"\nSão o mesmo instante? {utc == sp == tokyo}")

In [ ]:
# 🔴 Misturar naive e aware levanta erro
try:
    naive - utc
except TypeError as erro:
    print(f"❌ {erro}")
    print("\n💡 Isso é bom: o Python se recusa a fazer uma conta sem sentido.")
    print("   Escolha um lado — de preferência, aware em UTC.")

In [ ]:
# A forma correta de pegar "agora"
agora_errado = datetime.now()                      # ❌ naive, fuso da máquina
agora_certo = datetime.now(timezone.utc)           # ✅ aware, UTC
agora_local = datetime.now(ZoneInfo("America/Sao_Paulo"))   # ✅ aware, SP

print(f"❌ datetime.now()            : {agora_errado}  (tz={agora_errado.tzinfo})")
print(f"✅ datetime.now(timezone.utc): {agora_certo.isoformat()}")
print(f"✅ datetime.now(ZoneInfo(SP)): {agora_local.isoformat()}")
print()
print("💡 datetime.utcnow() está DEPRECIADO — ele devolve naive com hora UTC,")
print("   que é o pior dos dois mundos. Use datetime.now(timezone.utc).")

## 3. Parsing e formatação

| Direção | Método | Exemplo |
|---------|--------|---------|
| texto → datetime | `strptime` | `datetime.strptime("12/08/2026", "%d/%m/%Y")` |
| datetime → texto | `strftime` | `agora.strftime("%d/%m/%Y")` |
| ISO 8601 | `fromisoformat` / `isoformat` | ✅ prefira estes |

### Códigos de formato

| Código | Significa | Exemplo |
|--------|-----------|---------|
| `%Y` `%y` | Ano com 4 / 2 dígitos | `2026` / `26` |
| `%m` `%B` `%b` | Mês número / nome / abrev. | `08` / `August` / `Aug` |
| `%d` | Dia | `12` |
| `%H` `%M` `%S` | Hora / minuto / segundo | `14` `30` `45` |
| `%A` `%a` | Dia da semana | `Wednesday` / `Wed` |
| `%j` | Dia do ano | `224` |
| `%Z` `%z` | Fuso | `UTC` / `+0000` |

In [ ]:
texto_br = "12/08/2026 14:30"
texto_iso = "2026-08-12T14:30:00"

d1 = datetime.strptime(texto_br, "%d/%m/%Y %H:%M")
d2 = datetime.fromisoformat(texto_iso)

print(f"strptime      : {d1}")
print(f"fromisoformat : {d2}")
print(f"Iguais? {d1 == d2}")
print()
print("Formatando o mesmo datetime:")
for fmt, descricao in [
    ("%d/%m/%Y", "brasileiro"),
    ("%Y-%m-%d", "ISO (ordena como texto!)"),
    ("%d de %B de %Y", "por extenso"),
    ("%A, %d/%m", "com dia da semana"),
    ("%Y-%m", "mês (para agrupar)"),
    ("%H:%M:%S", "só hora"),
]:
    print(f"  {fmt:<18} {d1.strftime(fmt):<28} {descricao}")

print(f"\nisoformat(): {d1.isoformat()}")

> 🧭 **Sempre que puder, use ISO 8601 (`AAAA-MM-DD`).**
>
> - Ordena corretamente como **texto** (foi por isso que o M03 guardou datas assim no SQLite)
> - É inequívoco: `03/04/2026` é 3 de abril ou 4 de março? Depende do país. `2026-04-03` não depende de nada.
> - É o padrão de JSON, APIs e bancos de dados.

In [ ]:
# Parsing tolerante: várias entradas possíveis
def parse_data_flexivel(texto: str) -> date | None:
    """Tenta vários formatos comuns. Devolve None se nenhum servir."""
    formatos = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%Y/%m/%d", "%d.%m.%Y"]
    texto = texto.strip()
    for fmt in formatos:
        try:
            return datetime.strptime(texto, fmt).date()
        except ValueError:
            continue
    return None


entradas = ["2026-08-12", "12/08/2026", "12-08-2026", "2026/08/12",
            "12.08.2026", "ontem", "", "2026-02-31"]

for e in entradas:
    resultado = parse_data_flexivel(e)
    marca = "✅" if resultado else "❌"
    print(f"  {marca} {e!r:<16} → {resultado}")

> 💡 **`2026-02-31` foi rejeitada.** O `strptime` valida a data, não só o formato — 31 de fevereiro não existe. Isso é uma checagem gratuita que você ganha ao usar a biblioteca em vez de fatiar a string na mão.

## 4. `logging` — o fim do `print`

| | `print` | `logging` |
|---|---------|-----------|
| Destino | Só o terminal | Terminal, arquivo, rede, syslog... |
| Níveis | Nenhum | DEBUG, INFO, WARNING, ERROR, CRITICAL |
| Contexto | Manual | Timestamp, módulo, linha, thread |
| Ligar/desligar | Apagar linhas | Mudar um nível de configuração |
| Em produção | Perde-se | Persiste e é pesquisável |
| Formato | Texto solto | Estruturável (JSON) |

### Os cinco níveis

| Nível | Valor | Quando usar |
|-------|-------|-------------|
| `DEBUG` | 10 | Detalhe para depurar. Desligado em produção. |
| `INFO` | 20 | Marcos normais: "carga iniciada", "1.200 linhas processadas" |
| `WARNING` | 30 | Algo estranho, mas o programa continua |
| `ERROR` | 40 | Uma operação falhou |
| `CRITICAL` | 50 | O programa não pode continuar |

In [ ]:
import logging
import sys

# Configuração básica
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,          # força reconfiguração (importante em notebook!)
)

log = logging.getLogger("atlas")

log.debug("Conectando ao banco em dados/atlas.db")
log.info("Carga iniciada: vendas_jul2026.csv")
log.warning("Linha 42 rejeitada: quantidade inválida")
log.error("Falha ao gravar saida/relatorio.json")
log.critical("Banco de dados inacessível — abortando")

> ⚠️ **`force=True` é essencial em notebooks.** O `basicConfig` só age se ainda não houver handler configurado. Sem `force`, reconfigurar numa segunda célula simplesmente não funciona, e você fica achando que o logging está quebrado.

In [ ]:
# Loggers hierárquicos: um por módulo
log_leitura = logging.getLogger("atlas.leitura")
log_metricas = logging.getLogger("atlas.metricas")
log_saida = logging.getLogger("atlas.relatorios")

log_leitura.info("Lendo dados/brutos/vendas_jul2026.csv")
log_metricas.info("Agregando por cidade")
log_saida.info("Gravando saida/relatorio.txt")

# A hierarquia permite controlar por área
print()
logging.getLogger("atlas.metricas").setLevel(logging.WARNING)
log_metricas.info("esta mensagem NÃO aparece — nível elevado")
log_metricas.warning("esta aparece")
log_leitura.info("e o módulo de leitura continua em INFO")

> 💡 **A convenção é `logging.getLogger(__name__)`** no topo de cada módulo. Isso cria automaticamente a hierarquia `atlas`, `atlas.leitura`, `atlas.metricas` — e permite silenciar um módulo barulhento sem tocar nos outros.

In [ ]:
# Registrando exceções COM o traceback
def dividir(a, b):
    return a / b


try:
    dividir(10, 0)
except ZeroDivisionError:
    log.exception("Falha no cálculo do ticket médio")
    # log.exception() = log.error() + traceback completo.
    # Use SEMPRE dentro de um except.

In [ ]:
# Interpolação preguiçosa: passe os argumentos, não formate antes
registros = list(range(1500))

# ❌ Formata SEMPRE, mesmo se o nível estiver desligado
log.debug(f"Processados {len(registros)} registros de {sum(registros)} total")

# ✅ Só formata se o nível DEBUG estiver ativo
log.debug("Processados %d registros de %d total", len(registros), sum(registros))

print("\n💭 Com f-string, a formatação acontece antes da chamada — o custo")
print("   existe mesmo com DEBUG desligado. Com %s, o logging só formata")
print("   se for realmente emitir. Em laço quente, a diferença é real.")

### Handlers — para onde o log vai

Um logger pode ter **vários** handlers, cada um com seu nível e formato.

| Handler | Destino |
|---------|---------|
| `StreamHandler` | Terminal (stdout/stderr) |
| `FileHandler` | Arquivo |
| `RotatingFileHandler` | Arquivo com rotação por tamanho |
| `TimedRotatingFileHandler` | Rotação por tempo (diária, etc.) |
| `SMTPHandler` | E-mail (para CRITICAL) |
| `NullHandler` | Descarta — use em bibliotecas |

In [ ]:
from pathlib import Path
from logging.handlers import RotatingFileHandler

Path("logs_aula").mkdir(exist_ok=True)

# Logger dedicado, sem herdar a configuração global
logger = logging.getLogger("atlas.demo")
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
logger.propagate = False        # não repassa ao logger raiz

# Handler 1: console, só INFO+, formato enxuto
console = logging.StreamHandler(sys.stdout)
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter("%(levelname)-8s %(message)s"))

# Handler 2: arquivo, tudo, formato completo com rotação
arquivo = RotatingFileHandler(
    "logs_aula/atlas.log", maxBytes=100_000, backupCount=3, encoding="utf-8"
)
arquivo.setLevel(logging.DEBUG)
arquivo.setFormatter(logging.Formatter(
    "%(asctime)s | %(levelname)-8s | %(name)s | %(funcName)s:%(lineno)d | %(message)s"
))

logger.addHandler(console)
logger.addHandler(arquivo)

logger.debug("Detalhe técnico — só vai para o arquivo")
logger.info("Carga iniciada")
logger.warning("3 linhas rejeitadas")
logger.error("Falha ao conectar no banco")

print("\n── Conteúdo do arquivo de log ──")
print(Path("logs_aula/atlas.log").read_text(encoding="utf-8"))

### Logging estruturado (JSON)

Log em texto é ótimo para humano e péssimo para máquina. Quando os logs vão para uma ferramenta de observabilidade (Datadog, Grafana Loki, CloudWatch), **JSON** permite filtrar e agregar por campo.

In [ ]:
import json


class FormatadorJSON(logging.Formatter):
    """Emite cada registro como uma linha JSON."""

    def format(self, registro: logging.LogRecord) -> str:
        dados = {
            "timestamp": datetime.fromtimestamp(registro.created, timezone.utc).isoformat(),
            "nivel": registro.levelname,
            "logger": registro.name,
            "mensagem": registro.getMessage(),
            "modulo": registro.module,
            "funcao": registro.funcName,
            "linha": registro.lineno,
        }
        if registro.exc_info:
            dados["excecao"] = self.formatException(registro.exc_info)
        # Campos extras passados via extra={...}
        for chave, valor in registro.__dict__.items():
            if chave.startswith("ctx_"):
                dados[chave[4:]] = valor
        return json.dumps(dados, ensure_ascii=False)


log_json = logging.getLogger("atlas.json")
log_json.handlers.clear()
log_json.setLevel(logging.INFO)
log_json.propagate = False

h = logging.StreamHandler(sys.stdout)
h.setFormatter(FormatadorJSON())
log_json.addHandler(h)

log_json.info("Carga concluída", extra={
    "ctx_arquivo": "vendas_jul2026.csv",
    "ctx_linhas_validas": 1197,
    "ctx_linhas_rejeitadas": 3,
    "ctx_duracao_ms": 842,
})

try:
    1 / 0
except ZeroDivisionError:
    log_json.exception("Erro no cálculo", extra={"ctx_etapa": "ticket_medio"})

> 💭 **Por que isso importa?** Com logs em JSON, a pergunta *"quantas cargas falharam nas últimas 24h, por arquivo?"* vira uma consulta. Com logs em texto solto, vira `grep` e sofrimento.
>
> No **Módulo 09** você vai enviar esses logs para uma ferramenta de monitoramento. O formato que você escolher aqui determina o que será possível perguntar lá.

## 5. Context managers

O `with` garante que a limpeza aconteça — **mesmo se der exceção no meio**.

Você já usa: `with open(...)`. Agora vai aprender a escrever os seus.

O protocolo tem dois métodos:

- `__enter__()` — executa na entrada; o retorno vai para o `as`
- `__exit__(tipo, valor, traceback)` — executa na saída, **sempre**

In [ ]:
import time


class Cronometro:
    """Mede o tempo de um bloco."""

    def __init__(self, nome="bloco"):
        self.nome = nome

    def __enter__(self):
        self.inicio = time.perf_counter()
        print(f"⏱  iniciando: {self.nome}")
        return self                  # vai para o 'as'

    def __exit__(self, tipo_exc, valor_exc, traceback):
        self.duracao = time.perf_counter() - self.inicio
        estado = "❌ falhou" if tipo_exc else "✅ concluído"
        print(f"⏱  {estado}: {self.nome} em {self.duracao*1000:.2f} ms")
        return False                 # False = NÃO suprime a exceção


with Cronometro("soma pesada") as c:
    total = sum(i ** 2 for i in range(1_000_000))

print(f"resultado: {total:,}")
print(f"duração acessível depois: {c.duracao*1000:.2f} ms")

In [ ]:
# O __exit__ roda MESMO com exceção
try:
    with Cronometro("operação que falha"):
        time.sleep(0.05)
        raise ValueError("algo deu errado no meio")
except ValueError as erro:
    print(f"\nExceção propagou normalmente: {erro}")

> 💡 **O retorno do `__exit__` importa.** `False` (ou `None`) deixa a exceção propagar — é o que você quer em 99% dos casos. `True` **engole** a exceção, o que raramente é correto e deve ser feito com muita consciência.

In [ ]:
# Context manager que gerencia recurso de verdade
import sqlite3


class Banco:
    """Conexão com commit/rollback automático."""

    def __init__(self, caminho=":memory:"):
        self.caminho = caminho
        self.conexao = None

    def __enter__(self):
        self.conexao = sqlite3.connect(self.caminho)
        self.conexao.execute("PRAGMA foreign_keys = ON")
        print("🔌 conexão aberta")
        return self.conexao

    def __exit__(self, tipo_exc, valor_exc, tb):
        if tipo_exc is None:
            self.conexao.commit()
            print("💾 commit")
        else:
            self.conexao.rollback()
            print(f"↩️  rollback ({tipo_exc.__name__})")
        self.conexao.close()
        print("🔌 conexão fechada")
        return False


with Banco() as conexao:
    conexao.execute("CREATE TABLE t (id INTEGER PRIMARY KEY, v TEXT)")
    conexao.execute("INSERT INTO t (v) VALUES ('ok')")
    print("   linhas:", conexao.execute("SELECT COUNT(*) FROM t").fetchone()[0])

print()
try:
    with Banco() as conexao:
        conexao.execute("CREATE TABLE t (id INTEGER PRIMARY KEY)")
        conexao.execute("INSERT INTO inexistente VALUES (1)")
except sqlite3.OperationalError as erro:
    print(f"   erro capturado: {erro}")

## 6. `contextlib` — atalhos

| Ferramenta | Faz |
|------------|-----|
| `@contextmanager` | Cria context manager a partir de um **gerador** |
| `suppress(Exc)` | Ignora exceções específicas |
| `redirect_stdout(f)` | Redireciona a saída |
| `ExitStack` | Combina N context managers dinamicamente |
| `closing(obj)` | Chama `.close()` na saída |
| `nullcontext()` | Context manager que não faz nada (útil como padrão) |

In [ ]:
from contextlib import contextmanager, suppress, redirect_stdout, ExitStack, nullcontext


@contextmanager
def cronometro(nome="bloco"):
    """A mesma coisa da classe Cronometro, em 8 linhas.

    Tudo antes do yield = __enter__
    Tudo depois        = __exit__
    O try/finally garante a limpeza mesmo com exceção.
    """
    inicio = time.perf_counter()
    print(f"⏱  {nome}...")
    try:
        yield                     # aqui roda o corpo do with
    finally:
        print(f"⏱  {nome}: {(time.perf_counter()-inicio)*1000:.2f} ms")


with cronometro("comprehension"):
    _ = [i ** 2 for i in range(500_000)]

In [ ]:
# @contextmanager com valor e tratamento de erro
@contextmanager
def etapa(nome, logger=None):
    """Marca uma etapa do pipeline, com log de início, fim e falha."""
    registro = {"nome": nome, "sucesso": None, "duracao_ms": 0}
    inicio = time.perf_counter()
    (logger or log).info("▶ %s", nome)
    try:
        yield registro            # o corpo pode escrever aqui
        registro["sucesso"] = True
    except Exception as erro:
        registro["sucesso"] = False
        registro["erro"] = str(erro)
        (logger or log).error("✖ %s falhou: %s", nome, erro)
        raise
    finally:
        registro["duracao_ms"] = round((time.perf_counter() - inicio) * 1000, 2)
        if registro["sucesso"]:
            (logger or log).info("✔ %s (%s ms)", nome, registro["duracao_ms"])


logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)

with etapa("Leitura do CSV") as e:
    time.sleep(0.03)
    e["linhas"] = 1200

print(f"→ registro da etapa: {e}")

In [ ]:
# suppress: mais limpo que try/except/pass
from pathlib import Path

caminho = Path("arquivo_que_nao_existe.txt")

# ❌ verboso
try:
    caminho.unlink()
except FileNotFoundError:
    pass

# ✅ intenção explícita
with suppress(FileNotFoundError):
    caminho.unlink()

print("✅ Nenhum erro nos dois casos")
print("💡 suppress diz 'eu SEI que isso pode falhar e não me importo'.")
print("   try/except/pass diz a mesma coisa, mas parece descuido.")

In [ ]:
# redirect_stdout: capturar saída de código que você não controla
import io

def funcao_de_terceiro():
    print("mensagem que eu não quero na tela")
    print("outra mensagem")
    return 42


buffer = io.StringIO()
with redirect_stdout(buffer):
    resultado = funcao_de_terceiro()

print(f"resultado: {resultado}")
print(f"saída capturada: {buffer.getvalue()!r}")

In [ ]:
# ExitStack: número variável de context managers
Path("dados_aula05").mkdir(exist_ok=True)
for i in range(3):
    Path(f"dados_aula05/parte_{i}.txt").write_text(f"conteúdo {i}\n", encoding="utf-8")

arquivos = sorted(Path("dados_aula05").glob("parte_*.txt"))

with ExitStack() as pilha:
    abertos = [pilha.enter_context(open(c, encoding="utf-8")) for c in arquivos]
    print(f"{len(abertos)} arquivos abertos simultaneamente")
    for f in abertos:
        print("  ", f.read().strip())
# todos fechados aqui, na ordem inversa

print("Todos fechados:", all(f.closed for f in abertos))

In [ ]:
# nullcontext: um context manager opcional sem duplicar código
def processar(dados, medir_tempo=False):
    contexto = cronometro("processamento") if medir_tempo else nullcontext()
    with contexto:
        return sum(dados)


print("sem medição:", processar(range(1000)))
print()
print("com medição:", processar(range(1000), medir_tempo=True))

## 🔧 Prática guiada — Observabilidade no Atlas

Vamos juntar tudo: logging configurado, context managers instrumentando cada etapa, e datas tratadas corretamente.

In [ ]:
%%writefile atlas_observabilidade.py
"""Infraestrutura de observabilidade do Atlas.

Três peças:
  • configurar_logging()  — handlers de console e arquivo
  • etapa()               — context manager que instrumenta e cronometra
  • Execucao              — agrega o resultado de todas as etapas
"""

from __future__ import annotations

import json
import logging
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from logging.handlers import RotatingFileHandler
from pathlib import Path


# ═══════════════════════════════════════════════════════════════
#  Formatação
# ═══════════════════════════════════════════════════════════════

class FormatadorJSON(logging.Formatter):
    """Uma linha JSON por registro — pronto para ferramenta de log."""

    CAMPOS_PADRAO = set(logging.LogRecord("", 0, "", 0, "", (), None).__dict__)

    def format(self, r: logging.LogRecord) -> str:
        dados = {
            "ts": datetime.fromtimestamp(r.created, timezone.utc).isoformat(),
            "nivel": r.levelname,
            "logger": r.name,
            "msg": r.getMessage(),
            "origem": f"{r.module}.{r.funcName}:{r.lineno}",
        }
        if r.exc_info:
            dados["excecao"] = self.formatException(r.exc_info)
        # Qualquer extra={...} vira campo de primeira classe
        for chave, valor in r.__dict__.items():
            if chave not in self.CAMPOS_PADRAO and not chave.startswith("_"):
                dados[chave] = valor
        return json.dumps(dados, ensure_ascii=False, default=str)


class FormatadorConsole(logging.Formatter):
    """Legível para humano, com ícone por nível."""

    ICONES = {"DEBUG": "·", "INFO": "▸", "WARNING": "⚠", "ERROR": "✖", "CRITICAL": "🔥"}

    def format(self, r: logging.LogRecord) -> str:
        icone = self.ICONES.get(r.levelname, " ")
        hora = datetime.fromtimestamp(r.created).strftime("%H:%M:%S")
        return f"{hora} {icone} {r.getMessage()}"


# ═══════════════════════════════════════════════════════════════
#  Configuração
# ═══════════════════════════════════════════════════════════════

def configurar_logging(
    nivel_console: int = logging.INFO,
    nivel_arquivo: int = logging.DEBUG,
    pasta_logs: Path | str = "logs",
    nome: str = "atlas",
) -> logging.Logger:
    """Configura console (humano) e arquivo (JSON, rotativo)."""
    pasta = Path(pasta_logs)
    pasta.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(nome)
    logger.handlers.clear()
    logger.setLevel(min(nivel_console, nivel_arquivo))
    logger.propagate = False

    console = logging.StreamHandler(sys.stdout)
    console.setLevel(nivel_console)
    console.setFormatter(FormatadorConsole())
    logger.addHandler(console)

    arquivo = RotatingFileHandler(
        pasta / f"{nome}.jsonl", maxBytes=5_000_000, backupCount=5, encoding="utf-8"
    )
    arquivo.setLevel(nivel_arquivo)
    arquivo.setFormatter(FormatadorJSON())
    logger.addHandler(arquivo)

    return logger


# ═══════════════════════════════════════════════════════════════
#  Instrumentação
# ═══════════════════════════════════════════════════════════════

@dataclass
class ResultadoEtapa:
    nome: str
    inicio: datetime
    duracao_ms: float = 0.0
    sucesso: bool | None = None
    erro: str | None = None
    metricas: dict = field(default_factory=dict)


@dataclass
class Execucao:
    """Agrega o resultado de uma execução inteira."""
    id: str
    inicio: datetime
    etapas: list[ResultadoEtapa] = field(default_factory=list)

    @property
    def duracao_ms(self) -> float:
        return round(sum(e.duracao_ms for e in self.etapas), 2)

    @property
    def sucesso(self) -> bool:
        return all(e.sucesso for e in self.etapas)

    def resumo(self) -> str:
        linhas = [
            f"Execução {self.id}",
            f"Início   {self.inicio.isoformat()}",
            f"Duração  {self.duracao_ms:,.0f} ms",
            f"Status   {'✅ sucesso' if self.sucesso else '❌ com falhas'}",
            "",
            f"{'Etapa':<28}{'ms':>10}{'':>4}{'Métricas'}",
            "─" * 74,
        ]
        for e in self.etapas:
            marca = "✅" if e.sucesso else "❌"
            met = ", ".join(f"{k}={v}" for k, v in e.metricas.items())
            linhas.append(f"{e.nome:<28}{e.duracao_ms:>10,.1f}  {marca}  {met}")
            if e.erro:
                linhas.append(f"{'':<30}└─ {e.erro}")
        return "\n".join(linhas)


class Monitor:
    """Cria etapas instrumentadas e agrega o resultado."""

    def __init__(self, logger: logging.Logger, id_execucao: str | None = None):
        self.logger = logger
        agora = datetime.now(timezone.utc)
        self.execucao = Execucao(
            id=id_execucao or agora.strftime("%Y%m%d-%H%M%S"),
            inicio=agora,
        )

    @contextmanager
    def etapa(self, nome: str):
        resultado = ResultadoEtapa(nome=nome, inicio=datetime.now(timezone.utc))
        self.execucao.etapas.append(resultado)
        inicio = time.perf_counter()

        self.logger.info("▶ %s", nome, extra={"etapa": nome, "evento": "inicio"})
        try:
            yield resultado.metricas          # o corpo escreve métricas aqui
            resultado.sucesso = True
        except Exception as erro:
            resultado.sucesso = False
            resultado.erro = f"{type(erro).__name__}: {erro}"
            self.logger.exception("✖ %s", nome, extra={"etapa": nome, "evento": "erro"})
            raise
        finally:
            resultado.duracao_ms = round((time.perf_counter() - inicio) * 1000, 2)
            if resultado.sucesso:
                self.logger.info(
                    "✔ %s (%.0f ms)", nome, resultado.duracao_ms,
                    extra={"etapa": nome, "evento": "fim",
                           "duracao_ms": resultado.duracao_ms, **resultado.metricas},
                )

In [ ]:
import importlib
import atlas_observabilidade as obs
importlib.reload(obs)

import random
import shutil
from pathlib import Path

shutil.rmtree("logs_pratica", ignore_errors=True)
logger = obs.configurar_logging(pasta_logs="logs_pratica")
monitor = obs.Monitor(logger, id_execucao="demo-001")

rnd = random.Random(42)

# ── Pipeline instrumentado ──
with monitor.etapa("Leitura do CSV") as m:
    time.sleep(0.04)
    m["linhas_lidas"] = 200_000
    m["arquivo"] = "vendas_jul2026.csv"

with monitor.etapa("Validação") as m:
    time.sleep(0.06)
    m["validas"] = 199_412
    m["rejeitadas"] = 588

with monitor.etapa("Agregação por cidade") as m:
    time.sleep(0.03)
    m["cidades"] = 5
    m["faturamento"] = 1_344_533.60

with monitor.etapa("Gravação dos relatórios") as m:
    time.sleep(0.02)
    m["arquivos"] = 3

In [ ]:
# Uma etapa que falha — repare que a execução é registrada mesmo assim
try:
    with monitor.etapa("Envio por e-mail") as m:
        m["destinatarios"] = 4
        raise ConnectionError("servidor SMTP não respondeu")
except ConnectionError:
    pass

print()
print(monitor.execucao.resumo())

In [ ]:
# O que foi para o arquivo JSON
print("── logs_pratica/atlas.jsonl ──\n")
for linha in Path("logs_pratica/atlas.jsonl").read_text(encoding="utf-8").splitlines()[:4]:
    dados = json.loads(linha)
    print(json.dumps(dados, ensure_ascii=False, indent=2)[:400])
    print()

In [ ]:
# 💡 E agora o ganho: os logs viram DADOS consultáveis
registros = [json.loads(l) for l in
             Path("logs_pratica/atlas.jsonl").read_text(encoding="utf-8").splitlines()]

fins = [r for r in registros if r.get("evento") == "fim"]
erros = [r for r in registros if r["nivel"] == "ERROR"]

print(f"Total de registros : {len(registros)}")
print(f"Etapas concluídas  : {len(fins)}")
print(f"Erros              : {len(erros)}")
print(f"\nEtapa mais lenta   : ", end="")
if fins:
    mais_lenta = max(fins, key=lambda r: r.get("duracao_ms", 0))
    print(f"{mais_lenta['etapa']} ({mais_lenta['duracao_ms']:.0f} ms)")
print(f"Tempo total        : {sum(r.get('duracao_ms', 0) for r in fins):.0f} ms")
if erros:
    print(f"\nFalha registrada   : {erros[0]['msg']} — {erros[0].get('etapa')}")

> 💭 **Volte à dor do início.** *"O script rodou de madrugada e falhou. Não sei em que etapa, com qual arquivo, que hora."*
>
> Agora você tem: timestamp em UTC, nome da etapa, arquivo processado, contadores de cada fase, duração de cada uma, o traceback completo da falha — e tudo isso em JSON, que uma ferramenta consegue agregar.
>
> No **Módulo 09**, esse `.jsonl` vai para uma plataforma de observabilidade e você monta um painel. O trabalho está feito aqui.

## 📝 Exercícios

**E1.** Escreva `idade_em_dias(nascimento)` e `proximo_aniversario(nascimento)` que devolva a data do próximo aniversário a partir de hoje.

**E2.** Escreva `trimestre(d)` que devolva `"2026-Q3"` para uma data. Depois `datas_do_trimestre(ano, trimestre)` que devolva `(primeiro_dia, ultimo_dia)`.

**E3.** Escreva `dias_uteis_entre(inicio, fim, feriados)` que desconte fins de semana **e** uma lista de feriados.

**E4.** Demonstre o bug de fuso: crie um `datetime` naive representando 23:30 em São Paulo, converta ingenuamente para UTC assumindo que é UTC, e mostre que a data do relatório muda de dia.

**E5.** Escreva `converter_fuso(dt, de, para)` que aceite datetimes naive, aplique o fuso de origem e converta para o destino.

**E6.** Escreva um parser tolerante que aceite 6 formatos de data brasileiros e retorne sempre ISO. Inclua tratamento para entradas inválidas.

**E7.** Configure um logger com três handlers: console (INFO), arquivo geral (DEBUG) e arquivo só de erros (ERROR). Prove que cada mensagem vai para os lugares certos.

**E8.** Escreva um `logging.Filter` customizado que descarte mensagens contendo dados sensíveis (CPF, e-mail, cartão) ou os mascare.

**E9.** Escreva um decorador `@com_log(nivel=logging.INFO)` que registre entrada, saída, argumentos e duração de qualquer função.

**E10.** Escreva um context manager de classe `ArquivoTemporario` que crie um arquivo na entrada e o apague na saída, mesmo com exceção.

**E11.** Reescreva o exercício 10 usando `@contextmanager`. Compare as duas versões.

**E12.** Escreva um context manager `transacao(conexao)` que faça commit no sucesso e rollback na falha, com log em ambos os casos. Teste os dois caminhos.

**E13.** Use `ExitStack` para abrir uma quantidade variável de arquivos (definida em execução) e processá-los em conjunto.

**E14.** Escreva `@contextmanager` chamado `nivel_de_log(logger, nivel)` que mude temporariamente o nível de um logger e o restaure na saída.

**E15.** Estenda o `Monitor` da prática guiada para gravar o resumo da execução em `saida/execucoes.jsonl`, uma linha por execução. Depois escreva uma consulta que mostre a duração média de cada etapa nas últimas N execuções.

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

In [ ]:
# E7

In [ ]:
# E8

In [ ]:
# E9

In [ ]:
# E10

In [ ]:
# E11

In [ ]:
# E12

In [ ]:
# E13

In [ ]:
# E14

In [ ]:
# E15

## 📋 Cola de referência

```python
# ── datetime ──
from datetime import date, time, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

d = date(2026, 8, 12)
dt = datetime(2026, 8, 12, 14, 30)
delta = timedelta(days=30, hours=4)

data + delta          # → data
data - data           # → timedelta
delta.days  delta.total_seconds()
dt.weekday()          # 0=segunda
d.replace(day=1)      # primeiro dia do mês

# ⚠️ timedelta NÃO aceita months/years

# ── Fusos ──
datetime.now()                          # ❌ naive
datetime.now(timezone.utc)              # ✅ aware, UTC
datetime.now(ZoneInfo("America/Sao_Paulo"))
dt.astimezone(ZoneInfo("Asia/Tokyo"))
# ⚠️ datetime.utcnow() está DEPRECIADO
# 🧭 guarde em UTC, converta na apresentação

# ── Parsing / formatação ──
datetime.strptime("12/08/2026", "%d/%m/%Y")
dt.strftime("%d/%m/%Y %H:%M")
datetime.fromisoformat("2026-08-12T14:30:00")   # ✅ prefira
dt.isoformat()
# %Y %m %d %H %M %S %B %A %j %z

# ── logging ──
import logging
log = logging.getLogger(__name__)        # convenção por módulo

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    force=True,                          # ⚠️ essencial em notebook
)

log.debug/info/warning/error/critical("msg")
log.exception("msg")                     # error + traceback (dentro de except)
log.info("Processados %d de %d", a, b)   # ✅ preguiçoso, não f-string
log.info("msg", extra={"campo": valor})  # contexto estruturado

# Handlers
logging.StreamHandler(sys.stdout)
logging.FileHandler("app.log", encoding="utf-8")
RotatingFileHandler("app.log", maxBytes=5_000_000, backupCount=5)
handler.setLevel(...)  handler.setFormatter(...)
logger.addHandler(h)   logger.propagate = False

# ── Context manager (classe) ──
class CM:
    def __enter__(self):
        return valor                     # vai para o 'as'
    def __exit__(self, tipo, valor, tb):
        # limpeza — roda SEMPRE
        return False                     # False = não suprime a exceção

# ── Context manager (gerador) ──
from contextlib import contextmanager

@contextmanager
def cm(arg):
    recurso = abrir(arg)                 # = __enter__
    try:
        yield recurso
    finally:
        recurso.fechar()                 # = __exit__

# ── contextlib ──
from contextlib import suppress, redirect_stdout, ExitStack, closing, nullcontext
with suppress(FileNotFoundError): ...
with redirect_stdout(io.StringIO()) as buf: ...
with ExitStack() as pilha:
    fs = [pilha.enter_context(open(c)) for c in caminhos]
contexto = cm() if condicao else nullcontext()
```

## ✅ Checklist de saída

- [ ] Diferencio `date`, `datetime`, `time` e `timedelta`
- [ ] Sei que `data - data` dá `timedelta` e `data + timedelta` dá data
- [ ] Sei por que `timedelta` não tem meses, e como resolver
- [ ] **Sei a diferença entre datetime naive e aware**
- [ ] Uso `datetime.now(timezone.utc)`, nunca `utcnow()` nem `now()` sozinho
- [ ] Guardo em UTC e converto na apresentação
- [ ] Uso ISO 8601 e sei por que ele ordena como texto
- [ ] Faço parsing tolerante com `strptime` em vez de fatiar strings
- [ ] **Uso `logging` em vez de `print`** em código que vai rodar sozinho
- [ ] Sei quando usar cada um dos cinco níveis
- [ ] Uso `logging.getLogger(__name__)` por módulo
- [ ] Uso `log.exception()` dentro de `except`
- [ ] Passo argumentos ao logging em vez de f-string
- [ ] Sei configurar múltiplos handlers com níveis e formatos distintos
- [ ] Sei por que log estruturado (JSON) importa em produção
- [ ] Escrevo context managers de classe (`__enter__`/`__exit__`)
- [ ] Escrevo context managers com `@contextmanager` e `try/finally`
- [ ] Sei que o retorno do `__exit__` controla a supressão da exceção
- [ ] Conheço `suppress`, `redirect_stdout`, `ExitStack` e `nullcontext`

---

### ➡️ Próxima aula

**`04_06_Concorrencia.ipynb`** — Threads, processos, o GIL e `asyncio`. Quando fazer várias coisas ao mesmo tempo ajuda — e quando não ajuda em nada.